# Teste manual — etapa 6 v2 do bounding box cardíaco (âncora pela aorta)

Sobe uma imagem **aleatória** do dataset local (`./dataset`), roda `load_and_preprocess_image`
(etapas 1-4), detecta e segmenta a aorta ascendente (Hough Circle Transform + Level Set),
calcula a âncora (centróide da máscara da aorta) e compara com os limites do label coronário —
o mesmo fluxo de `process_case_for_aorta_anchor`, mas passo a passo, com visualização de cada
etapa intermediária (círculos detectados, máscara da aorta, âncora, cobertura do label).

Modelado em [check_preprocessing_and_centroid.ipynb](check_preprocessing_and_centroid.ipynb),
nos mesmos moldes (mesma seleção aleatória de imagem, mesmo bootstrap de ambiente).

Ver [docs/step6v2_aorta_anchor.md](../../docs/step6v2_aorta_anchor.md) para o racional completo
(por que a âncora mudou do LCC para a aorta, achados de cobertura em z, decisões em aberto antes
do lote completo).

**Aviso de custo**: detecção de círculos (Hough) + Level Set levam ~35s no total nesta máquina
sem GPU — normal, não é um erro se a célula demorar.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent,
    NOTEBOOK_CWD.parent.parent,
):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()
print(f"REPO_ROOT = {REPO_ROOT}")


In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np

from utils.processing.binary_operations import keep_largest_component
from utils.project.config import load_config_json
from utils.segmentation.aorta_localization import detect_aorta_circles
from utils.segmentation.aorta_segmentation import level_set_segmentation, remove_leaks_morphology
from utils.segmentation.bounding_box import (
    compute_bounding_box_bounds,
    compute_geometric_centroid,
)
from utils.segmentation.pipeline_preprocessing import load_and_preprocess_image
from utils.utils.nifti_io import load_raw_img_and_label
from utils.visualization.image_slices import (
    plot_mip_projection,
    plot_slices,
    visualize_circles_on_slices,
)

CONFIG_PATH = REPO_ROOT / "config" / "pipeline_config.json"
DATASET_ROOT = REPO_ROOT / "dataset"

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em {DATASET_ROOT}. Confirme o symlink/junction 'dataset' "
        "na raiz do repositório (ver .gitignore)."
    )


## 1. Escolher uma imagem aleatória do dataset local

Mesma busca recursiva usada em `check_preprocessing_and_centroid.ipynb` — o dataset está
organizado em subpastas por faixa de ID (`1-200`, `201-400`, ...).

In [ ]:
img_paths = sorted(DATASET_ROOT.glob("*/*.img.nii.gz"))
print(f"Total de imagens encontradas: {len(img_paths)}")

chosen_path = random.choice(img_paths)
IMG_ID = int(chosen_path.name.split(".")[0])
BASE_PATH = chosen_path.parent

print(f"Imagem escolhida: IMG_ID={IMG_ID}")
print(f"base_path={BASE_PATH}")


## 2. Rodar o pré-processamento (etapas 1-4)

`load_and_preprocess_image` encadeia downscale → threshold HU adaptativo → LCC por fatia.
`lcc_image` é a entrada usada pela detecção/segmentação da aorta a seguir.

In [ ]:
CONFIG = load_config_json(str(CONFIG_PATH), {})

image_data = load_and_preprocess_image(IMG_ID, str(BASE_PATH), CONFIG)
lcc_image = image_data["lcc_image"]
lcc_mask = image_data["lcc_mask"]
label = image_data["label"]
scaled_spacing = image_data["scaled_spacing"]
label_artery = label == 1

print(f"lcc_image: shape={lcc_image.shape} dtype={lcc_image.dtype}")
print(f"scaled_spacing (y, x, z) = {scaled_spacing}")
print(f"label: voxels_artéria={int(label_artery.sum())}")


## 3. Etapa 6 v2, parte 1 — Detecção da aorta (Hough Circle Transform)

`detect_aorta_circles` rastreia a aorta ascendente fatia a fatia, do topo do volume para baixo,
com tolerância de variação de raio/distância em mm. Mesma construção de parâmetros usada
internamente por `aorta_anchor.segment_aorta`.

In [ ]:
circle_config = CONFIG["CIRCLE_DETECTION"]
hough_radii = np.arange(
    circle_config["radii_start_px"],
    circle_config["radii_end_px"],
    circle_config.get("radius_step_px", 1),
)
pixel_spacing = (scaled_spacing[0] + scaled_spacing[1]) / 2.0

t0 = time.perf_counter()
detected_circles = detect_aorta_circles(
    lcc_image,
    hough_radii,
    pixel_spacing,
    tol_radius_mm=circle_config["tol_radius_mm"],
    tol_distance_mm=circle_config["tol_distance_mm"],
    quadrant_offset=tuple(circle_config["quadrant_offset"]),
    max_slice_miss_threshold=circle_config["max_slice_miss_threshold"],
    neighbor_distance_threshold=circle_config["neighbor_distance_threshold"],
    total_num_peaks_initial=circle_config["total_num_peaks_initial"],
    total_num_peaks=circle_config["total_num_peaks"],
    canny_sigma=circle_config["canny_sigma"],
    use_local_roi=circle_config.get("use_local_roi", True),
    local_roi_padding=circle_config.get("local_roi_padding", 20),
    out_of_tolerance_as_miss=circle_config.get("out_of_tolerance_as_miss", False),
    interpolate_missed_circles=circle_config.get("interpolate_missed_circles", True),
    candidate_selection_strategy=circle_config.get("candidate_selection_strategy", "closest"),
    use_gpu=bool(CONFIG.get("USE_GPU", False)),
    verbose=False,
)
detect_seconds = time.perf_counter() - t0

print(f"Tempo de detecção: {detect_seconds:.1f}s")
print(f"Círculos detectados: {len(detected_circles)}")

if not detected_circles:
    raise RuntimeError(
        f"Nenhuma aorta detectada para IMG_ID={IMG_ID} — rode a célula 'pick-image' de novo "
        "para sortear outro caso (falha de detecção é esperada em ~1% dos casos)."
    )

z_values = [c["slice_index"] for c in detected_circles]
x_values = [c["center_x"] for c in detected_circles]
y_values = [c["center_y"] for c in detected_circles]
print(f"z (fatias): {min(z_values)} - {max(z_values)} de {lcc_image.shape[2]}")
print(f"center_x: {min(x_values):.1f} - {max(x_values):.1f}")
print(f"center_y: {min(y_values):.1f} - {max(y_values):.1f}")


## 4. Visualização — círculos detectados

Sobrepõe os círculos rastreados em fatias amostradas, para inspeção qualitativa (devem seguir a
aorta ascendente, não outra estrutura circular do tórax).

In [ ]:
visualize_circles_on_slices(lcc_image, detected_circles, num_samples=6)


## 5. Etapa 6 v2, parte 2 — Segmentação da aorta (Level Set)

`level_set_segmentation` refina os círculos detectados numa máscara 3D via contorno ativo
geodésico morfológico (MGAC); `remove_leaks_morphology` + `keep_largest_component` limpam a
máscara. Mesma sequência de `aorta_anchor.segment_aorta`.

In [ ]:
level_set_config = CONFIG["LEVEL_SET"]
use_gpu = bool(CONFIG.get("USE_GPU", False))

t0 = time.perf_counter()
aorta_mask = level_set_segmentation(
    lcc_image,
    detected_circles,
    radius_reduction_factor=level_set_config["radius_reduction_factor"],
    num_iter=level_set_config["num_iter"],
    balloon=level_set_config["balloon"],
    smoothing=level_set_config["smoothing"],
    use_gpu=use_gpu,
)
aorta_mask = remove_leaks_morphology(
    aorta_mask, radius=level_set_config["leak_removal_radius"], use_gpu=use_gpu
)
aorta_mask = keep_largest_component(aorta_mask, gpu=use_gpu).astype(np.uint8)
segment_seconds = time.perf_counter() - t0

print(f"Tempo de segmentação (Level Set + limpeza): {segment_seconds:.1f}s")
print(f"Voxels da máscara da aorta: {int(aorta_mask.sum())}")

aorta_zs = np.nonzero(aorta_mask)[2]
print(f"z (fatias) da máscara final: {aorta_zs.min()} - {aorta_zs.max()} de {aorta_mask.shape[2]}")


## 6. Âncora — centróide geométrico da máscara da aorta

Mesma função `compute_geometric_centroid` já usada na etapa 5 (antes aplicada ao `lcc_mask`,
agora à máscara da aorta) — reaproveitada sem alteração.

In [ ]:
anchor = compute_geometric_centroid(aorta_mask)
anchor_y, anchor_x, anchor_z = anchor

print(f"Âncora (y, x, z) = ({anchor_y:.2f}, {anchor_x:.2f}, {anchor_z:.2f})")

assert 0 <= anchor_y < aorta_mask.shape[0]
assert 0 <= anchor_x < aorta_mask.shape[1]
assert 0 <= anchor_z < aorta_mask.shape[2]
print("Âncora dentro dos limites do volume: OK")


## 7. Âncora sobreposta na fatia correspondente

Marca `(x, y)` da âncora na fatia axial mais próxima de `z`, para conferir visualmente que cai
dentro/perto da aorta (e não deslocada por uma troca de eixos).

In [ ]:
z_slice = int(round(anchor_z))

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(lcc_image[:, :, z_slice], cmap="gray")
ax.contour(aorta_mask[:, :, z_slice], colors="yellow", linewidths=1)
ax.plot(anchor_x, anchor_y, "r+", markersize=16, markeredgewidth=2)
ax.set_title(
    f"IMG_ID={IMG_ID} — fatia z={z_slice} | âncora (y={anchor_y:.1f}, x={anchor_x:.1f})"
)
ax.axis("off")
plt.show()


## 8. Visualização — MIP da máscara da aorta vs. label coronário

Compara a projeção de intensidade máxima (MIP axial) da máscara da aorta segmentada com a do
label real de artéria coronária — mostra visualmente que a aorta ocupa a porção superior do
volume, enquanto a árvore coronária se estende mais abaixo (achado da sessão de calibração).

In [ ]:
plot_mip_projection(
    aorta_mask.astype(np.float32),
    title=f"IMG_ID={IMG_ID} — máscara da aorta (Level Set)",
    views=["axial"],
)

if label_artery.sum() > 0:
    plot_mip_projection(
        label_artery.astype(np.float32),
        title=f"IMG_ID={IMG_ID} — label artéria coronária",
        views=["axial"],
    )
else:
    print("Label sem voxels de artéria após downscale (raro, mas possível em labels muito finos).")


## 9. Cobertura em z — âncora vs. label coronário

Compara a faixa de fatias (z) cobertas pela aorta com a faixa ocupada pelo label coronário real,
e os offsets (`label - âncora`) por eixo — a mesma grandeza que `calibrate_bbox_offsets` agrega
entre casos (min/max) para calibrar o tamanho fixo do bbox.

In [ ]:
label_min, label_max = compute_bounding_box_bounds(label_artery)

axes_names = ("y", "x", "z")
anchor_coords = (anchor_y, anchor_x, anchor_z)

print(f"{'eixo':<5}{'label_min':>10}{'label_max':>10}{'âncora':>10}{'offset_min':>12}{'offset_max':>12}")
for name, lo, hi, a in zip(axes_names, label_min, label_max, anchor_coords):
    print(f"{name:<5}{lo:>10.1f}{hi:>10.1f}{a:>10.1f}{lo - a:>12.1f}{hi - a:>12.1f}")

fig, ax = plt.subplots(figsize=(8, 2))
ax.barh(0, aorta_zs.max() - aorta_zs.min(), left=aorta_zs.min(), height=0.3, label="aorta (z)")
ax.barh(
    1,
    label_max[2] - label_min[2],
    left=label_min[2],
    height=0.3,
    label="label coronário (z)",
)
ax.axvline(anchor_z, color="red", linestyle="--", label="âncora (z)")
ax.set_yticks([0, 1])
ax.set_yticklabels(["aorta", "label"])
ax.set_xlabel("fatia (z, resolução reduzida)")
ax.set_title(f"IMG_ID={IMG_ID} — cobertura em z: aorta vs. label coronário")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


## 10. Bbox aproximado desta imagem + Rv estimado

⚠️ **Isto não é o bbox fixo final da tese.** O bbox fixo (etapas 6-8) precisa ser calibrado sobre
o split de **treino inteiro** (`calibrate_bbox_offsets`, ver
[docs/step6v2_aorta_anchor.md](../../docs/step6v2_aorta_anchor.md)) — usa o offset mais
conservador (min/max) entre muitos casos, então tende a ser **maior** que o mostrado aqui.

Esta célula usa só o label **desta imagem** para aproximar o bbox — equivalente a
`bbox_min = âncora + offset_min` e `bbox_max = âncora + offset_max` da célula anterior, que por
construção coincidem com `label_min`/`label_max` (o offset foi calculado a partir do próprio
label). É uma ilustração otimista (o menor bbox que cobriria só este caso), útil para visualizar
a geometria e ter uma primeira noção de `Rv`, mas o `Rv` calibrado final tende a ser menor que o
estimado aqui (bbox real será maior para cobrir o pior caso do treino).

**Sobre a imagem:** o retângulo é o menor bbox que contém o *label* de artéria coronária — não o coração inteiro. É esperado (e é de onde vem o `Rv`) que tecido cardíaco visível na imagem fique fora do retângulo; o que precisa estar sempre dentro é o label em si, verificado explicitamente na célula seguinte.

In [ ]:
bbox_min = label_min
bbox_max = label_max
bbox_extent = bbox_max - bbox_min + 1

print(f"bbox aproximado (min) y,x,z = {tuple(bbox_min)}")
print(f"bbox aproximado (max) y,x,z = {tuple(bbox_max)}")
print(f"bbox extent (y, x, z)       = {tuple(bbox_extent)} voxels (resolução reduzida)")

# 6 fatias amostradas dentro da faixa em z do bbox.
sample_z = sorted(set(np.linspace(bbox_min[2], bbox_max[2], 6).round().astype(int).tolist()))

fig, axes = plt.subplots(1, len(sample_z), figsize=(4 * len(sample_z), 4.5))
if len(sample_z) == 1:
    axes = [axes]
for i, (ax, z) in enumerate(zip(axes, sample_z)):
    ax.imshow(lcc_image[:, :, z], cmap="gray")

    label_slice = label_artery[:, :, z]
    if label_slice.any():
        ys, xs = np.nonzero(label_slice)
        ax.scatter(
            xs, ys, s=6, c="red", label="artéria (label)" if i == 0 else None
        )

    rect = plt.Rectangle(
        (bbox_min[1], bbox_min[0]),
        bbox_extent[1],
        bbox_extent[0],
        fill=False,
        edgecolor="lime",
        linewidth=2,
        label="bbox aproximado" if i == 0 else None,
    )
    ax.add_patch(rect)
    ax.set_title(f"z={z}")
    ax.axis("off")

axes[0].legend(loc="upper right", fontsize=8, framealpha=0.7)
plt.suptitle(
    f"IMG_ID={IMG_ID} — bbox aproximado vs. voxels de artéria coronária (label) em 6 fatias",
    fontsize=13,
)
plt.tight_layout()
plt.show()

# Rv estimado: volume do bbox (aproximado) vs. volume da imagem original em resolução integral.
V_bbox_mm3 = float(np.prod(bbox_extent) * np.prod(scaled_spacing))

nii_img, _ = load_raw_img_and_label(str(chosen_path))
V_original_mm3 = float(np.prod(nii_img.shape) * np.prod(image_data["spacing"]))

Rv_estimado = 1.0 - (V_bbox_mm3 / V_original_mm3)

print()
print(f"Volume do bbox aproximado: {V_bbox_mm3 / 1000:,.1f} cm³")
print(f"Volume da imagem original: {V_original_mm3 / 1000:,.1f} cm³")
print(f"Rv estimado (só para este caso, não é o Rv final calibrado): {Rv_estimado:.2%}")


## 11. Verificação — todos os voxels do label estão dentro do bbox?

Checagem voxel a voxel (não só visual): para cada voxel de artéria coronária no label, confirma
se as três coordenadas `(y, x, z)` caem dentro de `[bbox_min, bbox_max]`. Como `bbox_min`/`bbox_max`
foram construídos como o bounding box exato deste label (célula anterior), o resultado esperado é
**100% de cobertura por construção** — a checagem serve para confirmar que não há bug de eixo/offset
na conta, não para "descobrir" uma falha (essa só apareceria ao aplicar offsets **calibrados no
treino**, etapa ainda não executada, a um caso de fora da calibração).

In [ ]:
label_coords = np.argwhere(label_artery)
within_y = (label_coords[:, 0] >= bbox_min[0]) & (label_coords[:, 0] <= bbox_max[0])
within_x = (label_coords[:, 1] >= bbox_min[1]) & (label_coords[:, 1] <= bbox_max[1])
within_z = (label_coords[:, 2] >= bbox_min[2]) & (label_coords[:, 2] <= bbox_max[2])
covered = within_y & within_x & within_z

num_total = len(label_coords)
num_covered = int(covered.sum())
print(
    f"Voxels de artéria coronária dentro do bbox: {num_covered}/{num_total} "
    f"({num_covered / num_total:.2%})"
)

if num_covered != num_total:
    missing = label_coords[~covered]
    raise AssertionError(
        f"{num_total - num_covered} voxels de artéria FORA do bbox — não deveria acontecer, já "
        f"que o bbox foi construído como o bbox exato deste label. Primeiros fora: {missing[:5]}"
    )

print("Cobertura 100% confirmada (esperado, por construção — ver nota da célula anterior).")


## 12. Resumo

Se todas as células acima rodaram sem erro e as visualizações fazem sentido anatomicamente
(círculos seguindo a aorta ascendente, máscara do Level Set compacta e sem vazamento, âncora
dentro da aorta, cobertura em z do label mais ampla que a da aorta), a etapa 6 v2 está
funcionando como esperado para esta imagem. Rode o notebook de novo para sortear outro caso —
lembre que a detecção falha em ~1% dos casos (ver célula 'detect-circles').

In [ ]:
print(f"IMG_ID testado: {IMG_ID}")
print(f"Círculos detectados: {len(detected_circles)} (tempo: {detect_seconds:.1f}s)")
print(f"Voxels da aorta: {int(aorta_mask.sum())} (tempo Level Set: {segment_seconds:.1f}s)")
print(f"Âncora (y, x, z): ({anchor_y:.2f}, {anchor_x:.2f}, {anchor_z:.2f})")
print(
    f"Offset em z necessário para cobrir o label: "
    f"[{label_min[2] - anchor_z:.1f}, {label_max[2] - anchor_z:.1f}]"
)
print("Todas as verificações (asserts) das células acima passaram.")
print(f"Rv estimado (bbox aproximado desta imagem): {Rv_estimado:.2%}")
